# Test Agent et Environment

- `Agent` : capteur actif avec dataset local
- `Environment` : médiateur de toutes les communications entre agents

**Flux de test:**
1. Charger un `MeteoDataset` (base de connaissances/ground truth)
2. Créer l'`Environment` 
3. Créer et enregistrer les agents
4. Initialiser les datasets locaux (depuis la base de connaissances)
5. Tester l'échange de données médié par l'environnement
6. Simuler un flux de données réel (injection ligne par ligne depuis le CSV)
7. Vérifier l'interaction entre agents après le flux

In [1]:
# Uncomment if dependencies are not installed
# %pip install -r requirements.txt

In [2]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath('.'))

from hybridiot import MeteoDataset, Agent, Environment
from datetime import datetime, timezone
import pandas as pd

## 1. Chargement de la base de connaissances

Le `MeteoDataset` contient les données d'initialisation. Il servira de
**ground truth** (données réelles d'OpenData) et de base de connaissances
pour les agents.

In [3]:
CSV_PATH = '../data/toulouse_meteo_live_jan2023(5stations).csv'

ds = MeteoDataset.from_csv(CSV_PATH)
print(f'Stations : {[s.id for s in ds.sensors]}')
print(f'Types    : {ds.data_types}')
print(f'Période  : {ds.start_time} -> {ds.end_time}')

  Sampling frequency detected: 15 min


Stations : ['Toulouse Avenue De Grande Bretagne', 'Toulouse Basso Cambo', 'Toulouse Montaudran', 'Toulouse Parc Jardin Des Plantes', 'Toulouse Soupetard']
Types    : ['temperature', 'humidite', 'pression', 'pluie', 'pluie_intensite_max']
Période  : 2023-01-01 00:00:00+00:00 -> 2023-01-31 23:45:00+00:00


## 2. Création de l'Environment

L'environnement est le seul canal de communication entre agents.
Un agent ne peut jamais communiquer directement avec un autre agent:
il demande à l'environnement, qui choisit les collaborateurs pertinents,
récupère les informations et les retourne au demandeur.

Le base_dataset est immuable: les données runtime vivent dans
les datasets locaux des agents.

In [4]:
env = Environment(base_dataset=ds)
print(env)

Environment(agents=0, base_dataset='toulouse_meteo_live_jan2023(5stations)')


## 3. Création et enregistrement des agents

Un agent = une station + un type de donnée.
On crée un agent température pour chaque station et on l'enregistre
dans l'environnement. Chaque agent reçoit automatiquement une
référence à l'environnement via register_agent().

In [6]:
for station in ds.sensors:
    agent = Agent(
        id         = station.id,
        position   = station.position,
        is_virtual = False,
        data_type  = 'temperature',
        frequency  = 15,
    )
    env.register_agent(agent)

print(env)
print(f'\nAgents enregistrés :')
for a in env.agents:
    print(f'  {a.id}  [{a.data_type}]  freq={a.frequency}min  env={a.environment is not None}')

Environment(agents=5, base_dataset='toulouse_meteo_live_jan2023(5stations)')

Agents enregistrés :
  Toulouse Avenue De Grande Bretagne  [temperature]  freq=15min  env=True
  Toulouse Basso Cambo  [temperature]  freq=15min  env=True
  Toulouse Montaudran  [temperature]  freq=15min  env=True
  Toulouse Parc Jardin Des Plantes  [temperature]  freq=15min  env=True
  Toulouse Soupetard  [temperature]  freq=15min  env=True


## 4. Initialisation des datasets locaux

Chaque agent charge ses données depuis le base_dataset (base de connaissances).
load_from_dataset() construit aussi les fault_periods automatiquement.

In [7]:
for agent in env.agents:
    agent.load_from_dataset(ds)

In [8]:
for agent in env.agents:
    print(f'{agent.id:<40} {len(agent.dataset):>6} valeurs   {len(agent.fault_periods):>3} fault_periods')

for agent in env.agents:
    print(f'\n{agent.id}:')
    for start, end in agent.fault_periods:
        print(f'  {start} -> {end}')

Toulouse Avenue De Grande Bretagne         2976 valeurs     1 fault_periods
Toulouse Basso Cambo                       2976 valeurs     0 fault_periods
Toulouse Montaudran                        2976 valeurs     3 fault_periods
Toulouse Parc Jardin Des Plantes           2976 valeurs     9 fault_periods
Toulouse Soupetard                         2976 valeurs   108 fault_periods

Toulouse Avenue De Grande Bretagne:
  2023-01-27 14:00:00+00:00 -> 2023-01-27 14:00:00+00:00

Toulouse Basso Cambo:

Toulouse Montaudran:
  2023-01-11 13:00:00+00:00 -> 2023-01-11 14:00:00+00:00
  2023-01-11 16:15:00+00:00 -> 2023-01-11 16:15:00+00:00
  2023-01-31 12:30:00+00:00 -> 2023-01-31 12:30:00+00:00

Toulouse Parc Jardin Des Plantes:
  2023-01-01 00:00:00+00:00 -> 2023-01-10 12:00:00+00:00
  2023-01-10 12:30:00+00:00 -> 2023-01-15 01:30:00+00:00
  2023-01-15 02:00:00+00:00 -> 2023-01-18 12:45:00+00:00
  2023-01-18 13:15:00+00:00 -> 2023-01-19 23:45:00+00:00
  2023-01-20 00:15:00+00:00 -> 2023-01-22 01:30

## 5. Échange de données à travers l'environnement

L'agent demande request_data_window() sans spécifier
à qui demander. L'environnement décide quels agents sont pertinents
via get_neighbours(), récupère leurs fenêtres, et les retourne
sous forme de dictionnaire {agent_id: DataWindow}.

In [9]:
agent_basso = env.get_agent('Toulouse Basso Cambo', data_type='temperature')
print(f'Agent demandeur : {agent_basso.id}\n')

start = datetime(2023, 1, 1,  0,  0, tzinfo=timezone.utc)
end   = datetime(2023, 1, 1, 23, 45, tzinfo=timezone.utc)

# L'agent demande des fenêtres mais c'est l'environnement qui choisit les collaborateurs
windows = agent_basso.request_data_window(
    start_time = start,
    end_time   = end,
)

print(f'Nombre de collaborateurs sélectionnés par l\'environnement : {len(windows)}')
print(f'\nFenêtres reçues :')
for agent_id, w in windows.items():
    print(f'  {agent_id:<40} shape={tuple(w.valeurs.shape)}  timesteps={w.n_timesteps}')

Agent demandeur : Toulouse Basso Cambo

Nombre de collaborateurs sélectionnés par l'environnement : 3

Fenêtres reçues :
  Toulouse Avenue De Grande Bretagne       shape=(1, 96, 1)  timesteps=96
  Toulouse Parc Jardin Des Plantes         shape=(1, 96, 1)  timesteps=96
  Toulouse Soupetard                       shape=(1, 96, 1)  timesteps=96


In [10]:
# Test : agent non enregistré dans l'environnement
fake = Agent(id='fantome', position=agent_basso.position, data_type='temperature')
print(f'Agent non enregistré, environment = {fake.environment}')
print(f'request_data_window() retourne : {fake.request_data_window()}')

Agent non enregistré, environment = None
request_data_window() retourne : {}


## 6 Simulation du flux de données (fonctionnement réel du système HybridIoT).

**Stratégie:**
1. Diviser le CSV en deux parties temporelles:
- première moitié = base de connaissances (chargée d'un coup dans l'Environment)
- seconde moitié = flux de données (injectée ligne par ligne aux agents)
2. Créer l'Environment et l'initialiser avec la première partie
3. Créer des agents vides et injecter la seconde partie progressivement avec add_value()
4. Observer l'évolution des fault_periods et des datasets locaux

In [11]:
# Lecture du CSV brut
with open(CSV_PATH, 'r') as f:
    first_line = f.readline()
sep = ';' if first_line.count(';') > first_line.count(',') else ','

df = pd.read_csv(CSV_PATH, sep=sep)
df['timestamp'] = pd.to_datetime(df['timestamp'], utc=True)
df = df.sort_values('timestamp').reset_index(drop=True)

print(f'CSV chargé : {len(df)} lignes')
print(f'Période    : {df["timestamp"].min()} -> {df["timestamp"].max()}')

CSV chargé : 11795 lignes
Période    : 2023-01-01 00:00:00+00:00 -> 2023-01-31 23:45:00+00:00


In [12]:
# Division temporelle: 
# première moitié = base 
# seconde moitié = flux
split_ts = df['timestamp'].quantile(0.5)
df_base   = df[df['timestamp'] <  split_ts]
df_stream = df[df['timestamp'] >= split_ts]

print(f'Base de connaissances : {len(df_base):>6} lignes  ({df_base["timestamp"].min()} -> {df_base["timestamp"].max()})')
print(f'Flux de données       : {len(df_stream):>6} lignes  ({df_stream["timestamp"].min()} -> {df_stream["timestamp"].max()})')

Base de connaissances :   5895 lignes  (2023-01-01 00:00:00+00:00 -> 2023-01-16 10:45:00+00:00)
Flux de données       :   5900 lignes  (2023-01-16 11:00:00+00:00 -> 2023-01-31 23:45:00+00:00)


In [13]:
# Nouvel environnement avec des agents vides
env_sim = Environment(base_dataset=df_base)

stations_uniques = df_base['station_name'].unique()
for station_name in stations_uniques:
    station = ds.get_sensor(station_name)
    if station is None:
        continue
    agent = Agent(
        id         = station.id,
        position   = station.position,
        data_type  = 'temperature',
        frequency  = 15,
    )
    env_sim.register_agent(agent)

print(f'{len(env_sim.agents)} agents créés (datasets vides)')
for a in env_sim.agents:
    print(f'  {a.id}  dataset={len(a.dataset)} valeurs')

5 agents créés (datasets vides)
  Toulouse Avenue De Grande Bretagne  dataset=0 valeurs
  Toulouse Montaudran  dataset=0 valeurs
  Toulouse Basso Cambo  dataset=0 valeurs
  Toulouse Soupetard  dataset=0 valeurs
  Toulouse Parc Jardin Des Plantes  dataset=0 valeurs


In [14]:
# Pré-indexation pour accélérer les recherches dans le loop
df_stream_indexed = df_stream.set_index(['timestamp', 'station_name'])

# Génération de tous les timestamps attendus (toutes les 15 min)
expected_timestamps = pd.date_range(
    start = df_stream['timestamp'].min(),
    end   = df_stream['timestamp'].max(),
    freq  = '15min',
    tz    = 'UTC'
)

import time
SLEEP_SECONDS = 0.001
n_injected    = 0

for ts in expected_timestamps:
    for agent in env_sim.agents:
        try:
            row = df_stream_indexed.loc[(ts, agent.id)]
            value = float(row['temperature']) if not pd.isna(row['temperature']) else float('nan')
        except KeyError:
            value = float('nan')   # timestamp manquant dans le CSV, NaN par défaut

        agent.add_value(value=value, timestamp=ts)
        n_injected += 1

    if n_injected % 500 == 0:
        print(f'  [{n_injected:>5} mesures injectées]')
        for a in env_sim.agents:
            print(f'    {a.id:<40} {len(a.dataset):>5} valeurs   {len(a.fault_periods):>3} fault_periods')

    time.sleep(SLEEP_SECONDS)

print(f'\nFlux terminé : {n_injected} mesures injectées')
print('\nÉtat final :')
for a in env_sim.agents:
    print(f'  {a.id:<40} {len(a.dataset):>6} valeurs   {len(a.fault_periods):>3} fault_periods')

  [  500 mesures injectées]
    Toulouse Avenue De Grande Bretagne         100 valeurs     0 fault_periods
    Toulouse Montaudran                        100 valeurs     0 fault_periods
    Toulouse Basso Cambo                       100 valeurs     0 fault_periods
    Toulouse Soupetard                         100 valeurs     3 fault_periods
    Toulouse Parc Jardin Des Plantes           100 valeurs     1 fault_periods
  [ 1000 mesures injectées]
    Toulouse Avenue De Grande Bretagne         200 valeurs     0 fault_periods
    Toulouse Montaudran                        200 valeurs     0 fault_periods
    Toulouse Basso Cambo                       200 valeurs     0 fault_periods
    Toulouse Soupetard                         200 valeurs     6 fault_periods
    Toulouse Parc Jardin Des Plantes           200 valeurs     1 fault_periods


  [ 1500 mesures injectées]
    Toulouse Avenue De Grande Bretagne         300 valeurs     0 fault_periods
    Toulouse Montaudran                        300 valeurs     0 fault_periods
    Toulouse Basso Cambo                       300 valeurs     0 fault_periods
    Toulouse Soupetard                         300 valeurs    10 fault_periods
    Toulouse Parc Jardin Des Plantes           300 valeurs     2 fault_periods
  [ 2000 mesures injectées]
    Toulouse Avenue De Grande Bretagne         400 valeurs     0 fault_periods
    Toulouse Montaudran                        400 valeurs     0 fault_periods
    Toulouse Basso Cambo                       400 valeurs     0 fault_periods
    Toulouse Soupetard                         400 valeurs    14 fault_periods
    Toulouse Parc Jardin Des Plantes           400 valeurs     3 fault_periods
  [ 2500 mesures injectées]
    Toulouse Avenue De Grande Bretagne         500 valeurs     0 fault_periods
    Toulouse Montaudran                        

## 7. Interaction entre agents après le flux

Après l'injection du flux, on vérifie que les agents peuvent toujours
échanger les données via l'environnement, y compris les nouvelles
mesures du flux.

In [15]:
agent_basso_sim = env_sim.get_agent('Toulouse Basso Cambo', data_type='temperature')
print(f'Agent demandeur : {agent_basso_sim.id}')
print(f'Taille dataset  : {len(agent_basso_sim.dataset)}')

# Fenêtre sur la période du flux (après le split)
windows_post = agent_basso_sim.request_data_window(
    start_time = split_ts.to_pydatetime(),
)

print(f'\nCollaborateurs sélectionnés : {len(windows_post)}')
for agent_id, w in windows_post.items():
    print(f'  {agent_id:<40} timesteps={w.n_timesteps}  ')

Agent demandeur : Toulouse Basso Cambo
Taille dataset  : 1492

Collaborateurs sélectionnés : 3
  Toulouse Avenue De Grande Bretagne       timesteps=1492  
  Toulouse Soupetard                       timesteps=1492  
  Toulouse Parc Jardin Des Plantes         timesteps=1492  


In [16]:
# Inspection d'une fenêtre reçue d'un collaborateur
first_collab = list(windows_post.keys())[1]
w = windows_post[first_collab]

print(f'Fenêtre de {first_collab} :')
print(f'  Shape     : {tuple(w.valeurs.shape)}')
print(f'  Intervalle: {w.interv_temps}')
print(f'  Premières valeurs du tenseur :')
print(f'  {w.valeurs.flatten()[:10]}')

Fenêtre de Toulouse Soupetard :
  Shape     : (1, 1492, 1)
  Intervalle: (Timestamp('2023-01-16 11:00:00+0000', tz='UTC'), Timestamp('2023-01-31 23:45:00+0000', tz='UTC'))
  Premières valeurs du tenseur :
  tensor([7.3000, 7.5000, 7.6000, 7.7000, 7.9000, 8.1000, 8.3000, 8.3000, 8.4000,
        8.4000])
